# Notebook 03 — Does Our Search Find the Right Records?

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

In notebook 02 we stored 337 records in Pinecone. Now we check: when we ask a question, does the search bring back
the **right** records? We use an **answer key** — questions where we already know the right answer — and count.

```text
answer key row                         vector DB                          compare
question + right ids        ──►   search(question) → top 5 ids   ──►   which of the 5 are right?
                                                                              │
                                                                              ▼
                                                     Hit@5 · Recall@5 · Precision@5 · MRR · nDCG@5
                                                                              │
                                                                              ▼
                                                     one row per question → averages → MLflow
```

```text
0 SETUP           connect, open the answer key, the search function
1 ONE QUESTION    search it, look at the top 5
2 THE METRICS     the 5 metrics, by hand, on that one question
3 ALL QUESTIONS   the same for every question in the answer key
4 AVERAGES        one score per sheet + the questions that failed
5 SAVE            MLflow
```

**You need:** notebook 02 run once, your `.env` file, and `ground_truth_ecom.xlsx` in the `docs` folder.

## 0 — Setup

Run these cells in order. Each one prints a ✅ when it worked.

In [ ]:
# Install the libraries (only needed once on your computer).
%pip install -q openai pinecone python-dotenv openpyxl mlflow

**0.1 — Connect** to OpenAI and Pinecone, with the same model, index and namespace as notebook 02.

In [ ]:
# Standard library modules for reading CSV files, doing math, handling
# environment variables, and working with dates and times.
import csv
import math
import os
from datetime import datetime

# Third-party libraries for reading Excel files and loading settings from .env.
import openpyxl
from dotenv import load_dotenv

# OpenAI provides the embedding model used to convert questions into vectors.
from openai import OpenAI

# Pinecone stores and searches the knowledge-base vectors.
from pinecone import Pinecone


# STEP 1 — Load environment variables from the .env file.
# This makes values such as OPENAI_API_KEY available through os.environ.
load_dotenv()


# STEP 2 — Check that both required API keys are present.
# Stop immediately with a clear error if either key is missing.
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")


# STEP 3 — Set the same configuration used by notebook 02.
# The embedding model must match the one used to create the stored vectors.
EMBED_MODEL = "text-embedding-3-small"

# Name of the Pinecone index that contains the knowledge-base records.
INDEX_NAME = "ecom-kb-tutorial"

# Namespace used to group the records within the index.
# Pass this namespace to Pinecone query or upsert calls when using it.
NAMESPACE = "ecom-kb"

# Maximum number of matching records to retrieve for each question.
TOP_K = 5


# STEP 4 — Connect to OpenAI and Pinecone.
# OpenAI() reads OPENAI_API_KEY from the environment automatically.
client = OpenAI()

# Create a Pinecone client with the API key, then open the specified index.
index = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
).Index(INDEX_NAME)


# Confirm that both clients are ready and show which index was opened.
print("✅ Connected to", INDEX_NAME)

**0.2 — Open the answer key.** Two small helpers: `read_sheet()` turns a sheet into a list of rows, and
`split_ids()` turns a cell holding several ids (one per line) into a Python list.

In [ ]:
# Open the Excel workbook in read-only mode because this code only reads it.
# This uses less memory than loading the workbook for editing.
answer_key = openpyxl.load_workbook(
    "./docs/ground_truth_ecom.xlsx",
    read_only=True,
)

In [ ]:
def read_sheet(sheet_name):
    """Read one worksheet and return its non-empty rows as dictionaries.

    The first row is treated as the column headers. For example, a row might
    become:
    {
        "qid": "T03",
        "question": "How much does a typical checkout bring in?",
        "right_id": "glossary::Average Order Value",
    }
    """
    # Read worksheet values only, without creating full cell objects.
    rows = answer_key[sheet_name].iter_rows(values_only=True)

    # The first worksheet row contains the column names.
    column_names = next(rows)

    records = []
    for values in rows:
        # Ignore rows that have no values in any column.
        if all(value is None for value in values):
            continue

        # Build a dictionary by pairing each column name with its row value.
        # zip() keeps each value associated with the header in the same position.
        record = {}
        for column_name, value in zip(column_names, values):
            record[column_name] = value

        records.append(record)

    # Return all non-empty worksheet rows as a list of dictionaries.
    return records

In [ ]:
def split_ids(cell):
    """Convert a cell containing newline-separated IDs into a list.

    For example:
    "glossary::Revenue
    column::ecom.orders.total_amount"

    becomes:
    ["glossary::Revenue", "column::ecom.orders.total_amount"]
    """
    # Return an empty list when the cell is blank or contains None.
    if not cell:
        return []

    ids = []

    # Convert the cell to text, then split it wherever there is a newline.
    for line in str(cell).split("\n"):
        # Remove spaces around each ID.
        line = line.strip()

        # Ignore blank lines and keep only lines containing an ID.
        if line:
            ids.append(line)

    return ids

In [ ]:
# Read each worksheet from the answer-key workbook into its own list.
# Each list contains one dictionary per non-empty worksheet row.

# Questions that describe a business term in different words.
glossary_questions = read_sheet("glossary_questions")

# Questions that describe a saved SQL example in different words.
example_questions = read_sheet("example_questions")

# Questions that ask about a relationship or join between two tables.
join_questions = read_sheet("join_questions")

# Real business questions that may have several correct records.
golden_questions = read_sheet("golden")


# Confirm that the workbook was read and show how many questions were loaded
# from each worksheet.
print("✅ Answer key opened")
print("   glossary questions:", len(glossary_questions))
print("   example questions: ", len(example_questions))
print("   join questions:    ", len(join_questions))
print("   golden questions:  ", len(golden_questions))

**0.3 — The search.** `search()` turns the question into numbers (an embedding) and asks Pinecone for the 5 closest
records. `tabs` limits the search to some tabs of the knowledge base — the same filters the agent uses.

In [ ]:
def search(question, tabs=None):
    """Search Pinecone and return the IDs of the TOP_K closest records, best first.

    Args:
        question: The text to search for.
        tabs: Optional list of worksheet names to search. For example,
              ["glossary", "golden"]. If omitted, search the whole namespace.

    Returns:
        A list of matching record IDs, ordered from closest match to least close.
    """
    # STEP 1 — Convert the question into an embedding vector.
    # The embedding model must match the model used for the stored records.
    # text-embedding-3-small produces 1,536 numbers by default.
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=[question],
    )
    question_vector = response.data[0].embedding

    # STEP 2 — Build a metadata filter only when worksheet names were provided.
    # This filter keeps records whose "sheet" metadata value matches one of tabs.
    # With no tabs (or an empty list), search without a sheet restriction.
    search_filter = None
    if tabs:
        search_filter = {"sheet": {"$in": tabs}}

    # STEP 3 — Ask Pinecone for the TOP_K nearest vectors to the question.
    # The namespace limits the search to the chosen group of records.
    # include_metadata=True asks Pinecone to include record metadata in results.
    results = index.query(
        vector=question_vector,
        top_k=TOP_K,
        namespace=NAMESPACE,
        include_metadata=True,
        filter=search_filter,
    )

    # STEP 4 — Collect the matching record IDs in Pinecone's returned order.
    # The first match is the closest result; this function does not return scores
    # or metadata, only the IDs.
    top_ids = []
    for match in results.matches:
        top_ids.append(match.id)

    return top_ids

In [ ]:
# Try it once.
print("✅ Search works. Top 5 for 'What was our GMV last month?':")
for record_id in search("What was our GMV last month?"):
    print("   ", record_id)

## 1 — One question

Take one row of the answer key: the question, and the record that **should** come back.

**Guess first:** *"How much does a typical checkout bring in?"* — the right answer is
`glossary::Average Order Value`. The question doesn't use those words. Will the search find it in the top 5?

In [ ]:
# Select one glossary question to try.
# Python list indexes start at 0, so index 2 selects the third question.
# Change the index to try a different question.
row = glossary_questions[2]

# Get the question text and its expected correct record ID or IDs.
question = row["question"]
right_ids = split_ids(row["right_id"])

# Search only the glossary and capability-card worksheets, where the answer
# is expected to be stored.
top_ids = search(question, tabs=["glossary", "capability_card"])

# Display the question and the expected answer IDs.
print("Question: ", question)
print("Right id: ", right_ids)
print()

# Show each search result in rank order. Add a marker when the result is one
# of the expected IDs so it's easy to see whether the correct record was found.
place = 1
for record_id in top_ids:
    if record_id in right_ids:
        print(f"  {place}. {record_id}   ✅ right")
    else:
        print(f"  {place}. {record_id}")

    # Advance the rank number for the next result.
    place += 1

That is the whole idea: **search, then compare the ids with the answer key.** Next we turn "how good was this?" into
numbers.

## 2 — The metrics, by hand

Write the top 5 as right / wrong. If the right answer came back in 5th place:

```text
place:     1      2      3      4      5
right?     ✗      ✗      ✗      ✗      ✓
```

| Metric | Question it answers | Formula | This example |
|---|---|---|---:|
| **Hit@5** | did we find **anything** right? | 1 if at least one ✓, else 0 | 1 |
| **Recall@5** | how many of the right records did we find? | ✓ found ÷ right records in the answer key | 1 ÷ 1 = 1.0 |
| **Precision@5** | how many of the 5 results are right? | ✓ found ÷ 5 | 1 ÷ 5 = 0.2 |
| **MRR** | how **high** was the first right one? | 1 ÷ its place | 1 ÷ 5 = 0.2 |
| **nDCG@5** | are the right ones near the **top**? | points for each ✓ (1 ÷ log2(place + 1)) ÷ the best possible points | 0.39 ÷ 1.0 = 0.39 |

**Two things to notice:**

- **Recall says "found it", MRR says "but low down".** Here the right answer was found (Recall 1.0) but only in 5th
  place (MRR 0.2). Both are true — that's why we look at more than one number.
- **Precision@5 can't go above 0.2 when there is only one right answer** (1 right out of 5 results). That's normal.
  Precision matters more for golden questions, which have several right records.

The next cell turns the table into one function and checks it on our question.

In [ ]:
def compare(top_ids, right_ids):
    """Compare search results with the answer key and return five metrics.

    Args:
        top_ids: IDs returned by search, ordered from best match to lowest.
        right_ids: IDs the answer key marks as correct.

    Returns:
        A dictionary containing Hit@5, Recall@5, Precision@5, MRR, and nDCG@5.
        This function assumes right_ids contains at least one ID.
    """
    # STEP 1 — Record whether each search result is in the answer key.
    # Each True or False stays in the same position as its result in top_ids.
    is_right = []
    for record_id in top_ids:
        is_right.append(record_id in right_ids)

    # Count how many correct records the search returned.
    found = is_right.count(True)

    # STEP 2 — Hit@5: 1 if at least one correct record appears in the results,
    # otherwise 0.
    if found > 0:
        hit = 1
    else:
        hit = 0

    # STEP 3 — Recall@5: the share of all correct answer-key records that
    # appeared in the search results.
    recall = found / len(right_ids)

    # STEP 4 — Precision@5: the share of the five requested results that are
    # correct. TOP_K is used as the denominator.
    precision = found / TOP_K

    # STEP 5 — MRR (Mean Reciprocal Rank for this one question):
    # Use the rank of the first correct result. A result at rank 1 scores 1.0,
    # at rank 2 scores 0.5, and so on. If none are correct, the score stays 0.
    mrr = 0
    place = 1
    for right in is_right:
        if right:
            mrr = 1 / place
            break
        place += 1

    # STEP 6 — Calculate the points earned by the returned results for nDCG@5.
    # A correct result earns less when it appears lower in the ranked list:
    # rank 1 earns 1.0, rank 2 about 0.63, rank 3 0.5, and rank 5 about 0.39.
    our_points = 0
    place = 1
    for right in is_right:
        if right:
            our_points += 1 / math.log2(place + 1)
        place += 1

    # Calculate the ideal score: the maximum possible points if all correct
    # records appeared first, up to TOP_K results.
    best_points = 0
    for place in range(1, min(len(right_ids), TOP_K) + 1):
        best_points += 1 / math.log2(place + 1)

    # Normalize the earned points against the ideal score.
    ndcg = our_points / best_points

    # Return all metrics together for convenient display or further analysis.
    return {
        "hit@5": hit,
        "recall@5": recall,
        "precision@5": precision,
        "mrr": mrr,
        "ndcg@5": ndcg,
    }


# Evaluate the search results for the question from chapter 1.
scores = compare(top_ids, right_ids)

# Print each metric to two decimal places.
for name, value in scores.items():
    print(f"  {name:12} {value:.2f}")

## 3 — All questions

Now the same two steps — **search, compare** — for every question of the answer key. Each sheet is searched in the
tabs where its right answers live:

| Sheet | Searched in | Right records per question |
|---|---|---|
| glossary | glossary + capability_card | 1 |
| examples | nlq_examples | 1 |
| joins | nlq_table + nlq_column + nlq_join | 1 |
| golden | **the whole knowledge base** — these questions need records from several tabs | several (`must_find`) |

This makes about 190 searches — about two minutes.

In [ ]:
# Which tabs to search for each sheet (None = the whole knowledge base).
SHEETS = [
    ("glossary", glossary_questions, ["glossary", "capability_card"], "right_id"),
    ("examples", example_questions, ["nlq_examples"], "right_id"),
    ("joins", join_questions, ["nlq_table", "nlq_column", "nlq_join"], "right_id"),
    ("golden", golden_questions, None, "must_find"),
]

results = []      # one row per question

for sheet_name, questions, tabs, right_column in SHEETS:
    for row in questions:
        right_ids = split_ids(row[right_column])

        # Some golden questions have nothing to find (they test "I can't answer that") — skip them here.
        if len(right_ids) == 0:
            continue

        # STEP 1 — search
        top_ids = search(row["question"], tabs)

        # STEP 2 — compare with the answer key
        scores = compare(top_ids, right_ids)

        # STEP 3 — keep one row for this question
        result = {"sheet": sheet_name, "qid": row["qid"], "question": row["question"],
                  "right_ids": " | ".join(right_ids), "top_5": " | ".join(top_ids)}
        result.update(scores)
        results.append(result)

    print(f"✅ {sheet_name} done")

print(f"\n{len(results)} questions checked")

In [ ]:
# Look at the first 5 rows of the results table.
for result in results[:5]:
    print(f"{result['qid']}  recall {result['recall@5']:.2f}  mrr {result['mrr']:.2f}   {result['question']}")

## 4 — Averages

One line per sheet: each metric averaged over all its questions. **1.0 = perfect.**

In [ ]:
METRICS = ["hit@5", "recall@5", "precision@5", "mrr", "ndcg@5"]


def averages(sheet_name):
    """Average each metric over the questions of one sheet."""
    rows = []
    for result in results:
        if result["sheet"] == sheet_name:
            rows.append(result)

    average = {}
    for metric in METRICS:
        total = 0
        for row in rows:
            total += row[metric]
        average[metric] = total / len(rows)
    return average, len(rows)


# Print the table: one line per sheet.
header = f"{'sheet':10}{'questions':>10}"
for metric in METRICS:
    header += f"{metric:>13}"
print(header)
AVERAGES = {}
for sheet_name, questions, tabs, right_column in SHEETS:
    average, count = averages(sheet_name)
    AVERAGES[sheet_name] = average
    line = f"{sheet_name:10}{count:>10}"
    for metric in METRICS:
        line += f"{average[metric]:>13.2f}"
    print(line)

**How to read it:**

- **Hit@5 / Recall@5** — how often the right record is found at all. The most important line.
- **MRR / nDCG@5** — much lower than Recall? Then the right records are found, but not near the top.
- **examples** usually scores highest (example questions sound like users); **joins** usually lowest (all join
  texts look alike: "Join A to B on …").
- **golden** is hardest: one search over the whole knowledge base must bring back records from different tabs.

Now the most useful part: **which questions failed?**

In [ ]:
# The questions where NOTHING right came back (Hit@5 = 0).
failed = []
for result in results:
    if result["hit@5"] == 0:
        failed.append(result)

print(f"{len(failed)} questions found nothing right. The first 10:\n")
for result in failed[:10]:
    print(f"{result['sheet']:9} {result['qid']}  {result['question']}")
    print(f"           should find: {result['right_ids']}")
    print(f"           got:         {result['top_5']}")
    print()

In [ ]:
# Save every question's result to a CSV you can open in Excel.
os.makedirs("eval_runs", exist_ok=True)
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
results_file = f"eval_runs/{run_name}.csv"

with open(results_file, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print("✅ Saved", results_file)

**When a question fails**, open its record in `kb_feed_ecom.xlsx` and compare it with the question:

| What you see | Fix |
|---|---|
| the record's text doesn't use the user's words | add them to `synonyms` / `analyst_synonyms` |
| the record's text is only a formula or code | rewrite `meaning` / `description` in plain words |
| the "wrong" result is really a better answer | fix the answer key |

Then reload in notebook 02 and run this notebook again. ⚠️ Never paste the test question itself into the Excel —
it would pass by copying, and prove nothing.

## 5 — Save the run to MLflow

MLflow keeps every run, so you can compare them side by side — for example before and after a fix.
See `README_MLflow.md` for how to open it in your browser.

```text
MLflow run
 ├── parameters   settings      embed_model, top_k
 ├── metrics      averages      glossary_recall_at_5, golden_mrr, …   (5 metrics × 4 sheets)
 └── artifact     a file        the CSV with one row per question
```

In [ ]:
import mlflow

# Runs are saved in mlflow.db, next to this notebook.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("vectordb-03-retrieval-eval")

with mlflow.start_run(run_name=run_name):

    # STEP 1 — parameters: the settings of this run
    mlflow.log_param("embed_model", EMBED_MODEL)
    mlflow.log_param("top_k", TOP_K)

    # STEP 2 — metrics: every average, named like "glossary_recall_at_5"
    #          (MLflow names can't contain "@", so "@" becomes "_at_")
    for sheet_name in AVERAGES:
        for metric, value in AVERAGES[sheet_name].items():
            mlflow.log_metric(f"{sheet_name}_{metric.replace('@', '_at_')}", value)

    # STEP 3 — artifact: the per-question CSV
    mlflow.log_artifact(results_file)

print("✅ Saved to MLflow as", run_name)
print("To look at it: mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001   (see README_MLflow.md)")

## What you learned

An **answer key** (questions + right ids) → the **same search** the agent uses → **compare** the ids → five
standard metrics per question → **averages** per sheet → the **failed questions** tell you what to fix → **MLflow**
keeps every run.

These are the same retrieval metrics used in any RAG project: Hit, Recall, Precision, MRR, nDCG.

## Quick check

1. The right answer came back in 2nd place. What are Hit@5, Recall@5 and MRR?
2. Why can Precision@5 never be more than 0.2 for a glossary question?
3. Recall is high but MRR is low. What does that mean?
4. A question fails. Do you fix the code or the Excel file?

<details><summary><b>Answers</b> (click)</summary>

1. Hit@5 = 1, Recall@5 = 1.0, MRR = 1 ÷ 2 = 0.5.
2. There is only one right record, and we look at 5 results: at best 1 of 5 is right.
3. The right records are found, but not near the top of the list.
4. Usually the Excel file: a missing word in the synonyms, or a meaning written only as a formula.

</details>

**Next:** Agent Version 3 uses this search as a tool — and then we also judge the **answers** (with an LLM judge).